## Libraries

In [0]:
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

## Load data

In [0]:
clean_data = spark.table('workspace.telco.bronze_data')

## Features

In [0]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
imputer = Imputer(inputCols=numeric_cols, outputCols=[c+"_imp" for c in numeric_cols])

categorical_cols = ["gender","Partner","Dependents","Contract","PaymentMethod","InternetService"]

indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep") for c in categorical_cols]
encoders = [OneHotEncoder(inputCol=c+"_idx", outputCol=c+"_ohe") for c in categorical_cols]

feature_cols = [c+"_imp" for c in numeric_cols] + [c+"_ohe" for c in categorical_cols]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

label_indexer = StringIndexer(inputCol="Churn", outputCol="label")

In [0]:
pipeline = Pipeline(stages=[imputer] + indexers + encoders + [assembler, label_indexer])
model = pipeline.fit(clean_data)

clean_data = model.transform(clean_data)

In [0]:
display(clean_data.limit(5))